# Part 2: Web Search — Inverted Index


**Classes implemented:**
- `MySet` — custom set with union and intersection
- `Position` — tuple of (PageEntry, wordIndex)
- `WordEntry` — all positions of a word across documents
- `PageIndex` — per-document word→positions map
- `MyHashTable` — maps word → WordEntry
- `InvertedPageIndex` — global inverted index across all pages
- `SearchEngine` — top-level engine with performAction()



## 1. Constants — Stop Words, Punctuation, Singular-Plural Map

In [1]:
import math
import os
import re

#  Stop words (connector words)
STOP_WORDS = {
    'a', 'an', 'the', 'they', 'these', 'this', 'for',
    'is', 'are', 'was', 'of', 'or', 'and', 'does', 'will', 'whose'
}

PUNCTUATION = set('{}[]<>=(). , ; \'"?#!-:')

# Singular-plural normalization
# Maps plural → singular (we normalize everything to singular form)
PLURAL_TO_SINGULAR = {
    'stacks': 'stack',
    'structures': 'structure',
    'applications': 'application',
}
# Build reverse map
NORMALIZE = {}
for plural, singular in PLURAL_TO_SINGULAR.items():
    NORMALIZE[plural]   = singular
    NORMALIZE[singular] = singular

def normalize_word(word: str) -> str:
    """
    Normalizes a word:
    1. Lowercase
    2. Map plural to singular if in the normalization table
    """
    word = word.lower()
    return NORMALIZE.get(word, word)


def remove_punctuation(text: str) -> str:
    """Replaces punctuation characters with spaces."""
    result = []
    for ch in text:
        if ch in PUNCTUATION:
            result.append(' ')
        else:
            result.append(ch)
    return ''.join(result)


def tokenize(text: str):
    """
    Tokenizes a webpage text into (word_index, normalized_word) pairs.

    Word indices are 1-based and count ALL tokens including stop words
    (stop words are counted in position but not stored).
    """
    text = remove_punctuation(text)
    raw_tokens = text.split()          # Split on whitespace

    result = []
    for idx, token in enumerate(raw_tokens, start=1):   # 1-based index
        norm = normalize_word(token)
        if norm not in STOP_WORDS:
            result.append((idx, norm))
    return result

print("Constants and helpers loaded.")
print("Stop words:", STOP_WORDS)
print("Normalization map:", NORMALIZE)

Constants and helpers loaded.
Stop words: {'these', 'of', 'will', 'for', 'or', 'an', 'whose', 'this', 'they', 'was', 'does', 'is', 'and', 'the', 'are', 'a'}
Normalization map: {'stacks': 'stack', 'stack': 'stack', 'structures': 'structure', 'structure': 'structure', 'applications': 'application', 'application': 'application'}


## 2. Class: `MySet`

A custom set supporting addElement, union, and intersection.

In [2]:
class MySet:

    def __init__(self):
        self._data = {}

    def addElement(self, element):
        """Add element to the set. Uses str(element) as the key."""
        self._data[str(element)] = element

    def union(self, otherSet: 'MySet') -> 'MySet':
        """Returns a new MySet that is the union of self and otherSet."""
        result = MySet()
        for v in self._data.values():
            result.addElement(v)
        for v in otherSet._data.values():
            result.addElement(v)
        return result

    def intersection(self, otherSet: 'MySet') -> 'MySet':
        """Returns a new MySet that is the intersection of self and otherSet."""
        result = MySet()
        for k, v in self._data.items():
            if k in otherSet._data:
                result.addElement(v)
        return result

    def getElements(self) -> list:
        """Returns a list of all elements in the set."""
        return list(self._data.values())

    def __len__(self):
        return len(self._data)

    def __repr__(self):
        return f"MySet({list(self._data.values())})"

## 3. Class: `Position`

Represents a (PageEntry, wordIndex) tuple — where a word occurs.

In [3]:
class Position:
    def __init__(self, page, wordIndex: int):
        self._page      = page
        self._wordIndex = wordIndex

    def getPageEntry(self):
        """Returns the PageEntry this position belongs to."""
        return self._page

    def getWordIndex(self) -> int:
        """Returns the 1-based word index."""
        return self._wordIndex

    def __repr__(self):
        return f"Position(page={self._page.getPageName()}, idx={self._wordIndex})"

    def __str__(self):
        return repr(self)

## 4. Class: `WordEntry`

Stores ALL positions across ALL documents where a particular word appears.
Also computes term frequency for TFIDF.

In [4]:
class WordEntry:
    """
    Stores all Position entries for a single word across all documents.
    Supports term-frequency computation for TFIDF scoring.
    """

    def __init__(self, word: str):
        self._word      = word
        self._positions = []   # List[Position]

    def addPosition(self, position: Position):
        """Add a single Position entry."""
        self._positions.append(position)

    def addPositions(self, positions: list):
        """Add multiple Position entries at once."""
        self._positions.extend(positions)

    def getAllPositionsForThisWord(self) -> list:
        """Returns the full list of Position entries for this word."""
        return list(self._positions)

    def getTermFrequency(self, page) -> float:
        """
        Computes the term frequency of this word in a given page:
            tf = (# occurrences in page) / (total words in page)
        """
        page_name = page.getPageName()
        count = sum(
            1 for pos in self._positions
            if pos.getPageEntry().getPageName() == page_name
        )
        total_words = page.getTotalWordCount()
        if total_words == 0:
            return 0.0
        return count / total_words

    def getPagesContainingWord(self) -> set:
        """Returns the set of page names that contain this word."""
        return {pos.getPageEntry().getPageName() for pos in self._positions}

    def getWord(self) -> str:
        return self._word

    def __repr__(self):
        return f"WordEntry(word='{self._word}', positions={len(self._positions)})"

## 5. Class: `MyHashTable`

Hash table mapping word → WordEntry. Uses Python's built-in `hash()` with chaining.

In [5]:
class MyHashTable:
    def __init__(self, capacity: int = 1024):
        self._capacity = capacity
        self._buckets  = [[] for _ in range(capacity)]  # List of chains

    def getHashIndex(self, word: str) -> int:
        return hash(word) % self._capacity

    def get(self, word: str):
        idx    = self.getHashIndex(word)
        bucket = self._buckets[idx]
        for stored_word, entry in bucket:
            if stored_word == word:
                return entry
        return None

    def addPositionsForWord(self, wordEntry: 'WordEntry'):
        word     = wordEntry.getWord()
        existing = self.get(word)

        if existing is None:
            # No entry yet — insert new
            idx = self.getHashIndex(word)
            self._buckets[idx].append((word, wordEntry))
        else:
            # Merge positions into existing entry
            existing.addPositions(wordEntry.getAllPositionsForThisWord())

    def getAllEntries(self) -> list:
        """Returns all WordEntry objects stored in the table."""
        result = []
        for bucket in self._buckets:
            for _, entry in bucket:
                result.append(entry)
        return result

    def __repr__(self):
        return f"MyHashTable(capacity={self._capacity}, entries={len(self.getAllEntries())})"

## 6. Class: `PageIndex`

Per-page index: maps each unique word in a document to its list of positions within that document.

In [6]:
class PageIndex:
    def __init__(self):
        self._index = {}

    def addPositionForWord(self, word: str, position: 'Position'):
        """
        Records that `word` occurs at `position` in this page.
        Creates a new entry if word is not yet in the index.
        """
        if word not in self._index:
            self._index[word] = []
        self._index[word].append(position)

    def getWordEntries(self) -> list:
        """
        Returns a list of WordEntry objects,
        one per unique word in this page.
        """
        entries = []
        for word, positions in self._index.items():
            we = WordEntry(word)
            we.addPositions(positions)
            entries.append(we)
        return entries

    def getPositionsForWord(self, word: str) -> list:
        """
        Returns list of Position objects for a word in this page.
        Returns empty list if word is not in this page.
        """
        return self._index.get(word, [])

    def containsWord(self, word: str) -> bool:
        return word in self._index

    def __repr__(self):
        return f"PageIndex({list(self._index.keys())})"

## 7. Class: `PageEntry`

Reads a webpage file and builds its `PageIndex`.

In [7]:
class PageEntry:
    """
    Represents a single webpage.
    On construction, reads the file from the webpages folder,
    tokenizes it, and builds a PageIndex.
    """

    # Folder where webpage files live — update if needed
    WEBPAGES_FOLDER = 'webpages'

    def __init__(self, pageName: str):
        """
        Reads the file named `pageName` from the webpages folder
        and builds the page index.

        """
        self._pageName   = pageName
        self._pageIndex  = PageIndex()
        self._totalWords = 0

        filepath = os.path.join(self.WEBPAGES_FOLDER, pageName)
        with open(filepath, 'r', encoding='utf-8') as f:
            text = f.read()

        # Count total tokens (including stop words) for TF denominator
        clean_text   = remove_punctuation(text)
        all_tokens   = clean_text.split()
        self._totalWords = len(all_tokens)

        # Build page index: only store non-stop words
        for word_idx, norm_word in tokenize(text):
            pos = Position(self, word_idx)
            self._pageIndex.addPositionForWord(norm_word, pos)

    def getPageName(self) -> str:
        """Returns the name of this webpage."""
        return self._pageName

    def getPageIndex(self) -> PageIndex:
        """Returns the PageIndex of this webpage."""
        return self._pageIndex

    def getTotalWordCount(self) -> int:
        """Returns total token count (including stop words), used for TF."""
        return self._totalWords

    def __repr__(self):
        return f"PageEntry('{self._pageName}')"

    def __str__(self):
        return self._pageName

## 8. Class: `InvertedPageIndex`

The global inverted index across all added pages.

In [8]:
class InvertedPageIndex:
    def __init__(self):
        self._hashTable = MyHashTable()
        self._pages     = {}

    def addPage(self, page: PageEntry):
        """
        Adds a new page to the inverted index.
        For each word in the page's index, merges its WordEntry
        into the global hash table.
        """
        self._pages[page.getPageName()] = page

        # For each unique word in this page, merge into global hash table
        for word_entry in page.getPageIndex().getWordEntries():
            self._hashTable.addPositionsForWord(word_entry)

    def getPagesWhichContainWord(self, word: str) -> MySet:

        result = MySet()
        entry  = self._hashTable.get(word)
        if entry is not None:
            for pos in entry.getAllPositionsForThisWord():
                result.addElement(pos.getPageEntry())
        return result

    def getPage(self, pageName: str):
        """Returns the PageEntry for a given page name, or None."""
        return self._pages.get(pageName, None)

    def getWordEntry(self, word: str):
        """Returns the WordEntry for a word, or None."""
        return self._hashTable.get(word)

    def getTotalPageCount(self) -> int:
        return len(self._pages)

    def __repr__(self):
        return f"InvertedPageIndex(pages={list(self._pages.keys())})"

## 9. Class: `SearchEngine`

Top-level engine. Parses and executes action strings.

In [16]:
class SearchEngine:

    def __init__(self):
        """Creates an empty InvertedPageIndex."""
        self._index = InvertedPageIndex()

    def performAction(self, actionMessage: str) -> str:
        """
        Main stub — parses and executes an action string.

        Supported actions:
          addPage <pageName>
          queryFindPagesWhichContainWord <word>
          queryFindPositionsOfWordInAPage <word> <pageName>

        """
        parts = actionMessage.strip().split()
        if not parts:
            return ''

        action = parts[0]

        # addPage
        if action == 'addPage':
            page_name = parts[1]
            page = PageEntry(page_name)
            self._index.addPage(page)
            # addPage produces no output (silent)
            return None

        # queryFindPagesWhichContainWord
        elif action == 'queryFindPagesWhichContainWord':
            raw_word  = parts[1]
            norm_word = normalize_word(raw_word)

            pages_set = self._index.getPagesWhichContainWord(norm_word)
            pages     = pages_set.getElements()

            if not pages:
                return f"No webpage contains word {raw_word}"
            else:
                # Return page names sorted alphabetically for consistency
                names = sorted([p.getPageName() for p in pages])
                return ', '.join(names)

        # queryFindPositionsOfWordInAPage
        elif action == 'queryFindPositionsOfWordInAPage':
            raw_word  = parts[1]
            page_name = parts[2]
            norm_word = normalize_word(raw_word)

            # Check if the page is in the database
            page = self._index.getPage(page_name)
            if page is None:
                return f"No webpage {page_name} found"

            # Check if the word exists in the page
            positions = page.getPageIndex().getPositionsForWord(norm_word)
            if not positions:
                return f"Webpage {page_name} does not contain word {raw_word}"

            # Return sorted word indices
            indices = sorted([p.getWordIndex() for p in positions])
            return ', '.join(map(str, indices))

        else:
            return f"Unknown action: {action}"

    def computeTFIDF(self, word: str, page_name: str) -> float:
        """
        Computes the TFIDF relevance score for a word in a page.
            TFIDF = TF(w, p) × IDF(w)
            TF    = freq(w, p) / total_words(p)
            IDF   = log(N / n_w)   where N = total pages, n_w = pages containing w

        """
        norm_word = normalize_word(word)
        page      = self._index.getPage(page_name)
        if page is None:
            return 0.0

        word_entry = self._index.getWordEntry(norm_word)
        if word_entry is None:
            return 0.0

        tf  = word_entry.getTermFrequency(page)
        N   = self._index.getTotalPageCount()
        n_w = len(word_entry.getPagesContainingWord())
        if n_w == 0:
            return 0.0

        idf = math.log(1 + (N / n_w))
        return tf * idf

print("All classes defined successfully.")

All classes defined successfully.


## 10. Setup — Copy Webpage Files into `webpages/` Folder

The `PageEntry` class reads files from a `webpages/` folder.

In [17]:
import os
import shutil

os.makedirs('webpages', exist_ok=True)

WEBPAGES = [
    'stack_datastructure_wiki',
    'stack_cprogramming',
    'stack_oracle',
    'stackoverflow',
    'stacklighting',
    'stackmagazine',
]

for wp in WEBPAGES:
    if os.path.exists(wp):  # files are uploaded in current directory
        shutil.copy(wp, os.path.join('webpages', wp))
        print(f"Copied: {wp}")
    else:
        print(f"[WARNING] Not found: {wp}")

print("\nWebpages folder ready:", os.listdir('webpages'))

Copied: stack_datastructure_wiki
Copied: stack_cprogramming
Copied: stack_oracle
Copied: stackoverflow
Copied: stacklighting
Copied: stackmagazine

Webpages folder ready: ['stackmagazine', 'stack_oracle', 'stack_cprogramming', 'stackoverflow', 'stacklighting', 'stack_datastructure_wiki']


## 11. Running `actions.txt` and Comparing with `answers.txt`

In [18]:
import os

def run_actions(actions_file, answers_file=None):
    engine = SearchEngine()

    with open(actions_file, 'r') as f:
        actions = [line.strip() for line in f if line.strip()]

    expected = []
    if answers_file and os.path.exists(answers_file):
        with open(answers_file, 'r') as f:
            expected = [line.strip() for line in f if line.strip()]

    print("ACTION".ljust(50), "OUTPUT".ljust(40), "EXPECTED".ljust(40), "MATCH")
    print("-" * 140)

    i = 0
    all_pass = True

    for action in actions:
        result = engine.performAction(action)

        if result is None:
            print(action.ljust(50), "(no output)")
            continue

        if i < len(expected):
            exp = expected[i]
            if result == exp:
                match = "YES"
            else:
                match = "NO"
                all_pass = False

            print(action.ljust(50), result.ljust(40), exp.ljust(40), match)
        else:
            print(action.ljust(50), result.ljust(40))

        i += 1

    print("-" * 140)

    if expected:
        if all_pass:
            print("All tests passed: YES")
        else:
            print("All tests passed: NO")


# run
run_actions(
    actions_file='actions.txt',
    answers_file='answers.txt'
)

ACTION                                             OUTPUT                                   EXPECTED                                 MATCH
--------------------------------------------------------------------------------------------------------------------------------------------
addPage stack_datastructure_wiki                   (no output)
queryFindPagesWhichContainWord delhi               No webpage contains word delhi           No webpage contains word delhi           YES
queryFindPagesWhichContainWord stack               stack_datastructure_wiki                 stack_datastructure_wiki                 YES
queryFindPagesWhichContainWord wikipedia           stack_datastructure_wiki                 stack_datastructure_wiki                 YES
queryFindPositionsOfWordInAPage magazines stack_datastructure_wiki Webpage stack_datastructure_wiki does not contain word magazines Webpage stack_datastructure_wiki does not contain word magazines YES
queryFindPagesWhichContainWord allain        

## 12. TFIDF Demo — Ranking Pages by Relevance

In [19]:
# Build a full engine with all pages for TFIDF demo
demo_engine = SearchEngine()
for wp in WEBPAGES:
    demo_engine.performAction(f'addPage {wp}')

# Rank all pages for query word 'stack'
query_word = 'push'
print(f"TFIDF scores for query word: '{query_word}'")
print("-" * 45)

scores = []
for wp in WEBPAGES:
    score = demo_engine.computeTFIDF(query_word, wp)
    scores.append((wp, score))

scores.sort(key=lambda x: x[1], reverse=True)
for page_name, score in scores:
    print(f"  {page_name:<35}  TFIDF = {score:.6f}")

print("\n(Higher TFIDF = more relevant for this query)")

TFIDF scores for query word: 'push'
---------------------------------------------
  stack_datastructure_wiki             TFIDF = 0.011916
  stack_oracle                         TFIDF = 0.007683
  stack_cprogramming                   TFIDF = 0.003544
  stackoverflow                        TFIDF = 0.000000
  stacklighting                        TFIDF = 0.000000
  stackmagazine                        TFIDF = 0.000000

(Higher TFIDF = more relevant for this query)
